<a href="https://colab.research.google.com/github/gt-cse-6040/bootcamp/blob/main/notebooks/sql/SQL1nb3_SQL_aggregate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Aggregate Functions

In [ ]:
!wget https://github.com/gt-cse-6040/bootcamp/raw/main/assets/data/NYC-311-2M_small.db

In [2]:
# create a connection to the database
import sqlite3 as db
import pandas as pd

# Connect to a database (or create one if it doesn't exist)
conn_nyc = db.connect('NYC-311-2M_small.db')

## Aggregate Functions

By definition, aggregate functions (like COUNT(), SUM(), and AVG()) take multiple rows of input data from a group and compress them into a single row of output.

However, there is a massive catch that constantly trips up database developers: With very few exceptions, aggregate functions completely ignore NULL values.

Failing to account for this default behavior is the #1 source of logic bugs and calculation errors in student queries. Because this concept is so vital to master, we are going to focus heavily on exactly how aggregations interpret missing data.


The following table summarizes some useful SQL aggregations:

| Aggregate Function       | Description                                              |
|--------------------------|----------------------------------------------------------|
| **COUNT( * )**           | Counts all rows in the group, including NULLs.           |
| **COUNT( value )**       | Counts only rows where that specific column is NOT NULL. |
| **AVG( value )**         | Computes the average of all non-NULL values.             |
| **MIN( value )**         | Returns the lowest non-NULL value.                       |
| **MAX( value )**         | Returns the highest non-NULL value.                      |
| **SUM (value )**         | Adds up all non-NULL numbers in the column.              |



**A few notes about COUNT() and SUM() concerning NULLs:**

COUNT(*) counts every single row in a group. It does not look at individual columns, meaning it counts rows even if they are entirely full of NULL values.

COUNT(column) only counts the number of times a valid, non-NULL value appears in that specific column.

SUM(column) adds up all non-NULL numbers in a column.

If all values in a column are integers, SUM() will return a clean integer. If the column contains decimals, it will return a decimal.

The Edge Case: If a column contains nothing but NULL values (or a WHERE clause filters your data down to zero rows), SUM() will return a blank NULL result because there were no numbers available to add together.

https://www.sqlite.org/lang_aggfunc.html

Let's look at some examples from the NYC 311 Calls database in Notebook 9.

For this exercise we have a subset of data, consisting of one month from 2014.

In [3]:
def count_all_rows():

    # count(*) returns all rows, including NULLS
    query_nulls = '''
                SELECT COUNT(*)
                FROM DATA
                '''
    return query_nulls

df_count_all_rows = pd.read_sql(count_all_rows(),conn_nyc)
display(df_count_all_rows)

,COUNT(*)
0,154374


In [4]:
def count_non_null_rows():

    # count(*) returns all rows, including NULLS
    # count only the non-NULL rows
    query_City = '''
                        SELECT COUNT(City)
                        FROM DATA
                        '''
    return query_City

df_count_non_null_rows = pd.read_sql(count_non_null_rows(),conn_nyc)
display(df_count_non_null_rows)

,COUNT(City)
0,145130


COUNT(*) returns the absolute total of 154,374 rows because it counts every record in the dataset, regardless of missing data.

COUNT(CITY) returns a lower number because it skips the ~9,000 rows where the city column is blank.

## Aggregations and String Case

Just like Python, SQLite is case-sensitive when comparing or grouping text data. This means SQLite views 'BROOKLYN', 'Brooklyn', and 'brooklyn' as three completely different values.

While some corporate database systems can be configured to ignore capitalization, SQLite enforces strict case sensitivity by default.

This behavior drastically impacts your aggregations. If your raw dataset has inconsistent capitalization, a GROUP BY clause will split what should be a single category into multiple separate rows. To prevent this, data analysts frequently use the built-in UPPER() or LOWER() functions to standardize text data before grouping it.

Now let's look at a simple GROUP BY

In [5]:
def simple_group_by():
    query_group = '''
                    SELECT City, COUNT(*)
                    FROM DATA
                    GROUP BY City
                    ORDER BY COUNT(*) DESC
                    LIMIT 5
                    '''
    return query_group

df_simple_group = pd.read_sql(simple_group_by(),conn_nyc)
display(df_simple_group)

,City,COUNT(*)
0,BROOKLYN,45664
1,NEW YORK,31667
2,BRONX,31372
3,None,9244
4,STATEN ISLAND,5222


As we can see, we simply returned the number of rows for each of the TOP 5 values in the `City` column.

Note that the `None` value is returned for `NULL` values in the `City` column.

#### Now let's look at string manipulation functions in SQL.

https://www.sqlitetutorial.net/sqlite-string-functions/

Some of the ones that we will use in this class are UPPER, LOWER, and SUBSTR.

The string functions generally work in the same manner as their Python equivalents, just check the documentation for the specific syntax.

Let's look at the UPPER function for some specific things that you should know.

In [ ]:
def upper_group_by():
    query_upper = '''
                    SELECT DISTINCT City, UPPER(City) as UPPER_CASE
                    FROM data
                    WHERE City != UPPER(City)
                    ORDER BY City
                    LIMIT 10
                    '''
    return query_upper

df_upper_group = pd.read_sql(upper_group_by(),conn_nyc)
display(df_upper_group)

In [ ]:
def upper_group_by_2():
    query_upper2 = '''
                    SELECT DISTINCT City, UPPER(City) as UPPER_CASE
                    FROM data
                    WHERE City = UPPER(City)
                    ORDER BY City
                    LIMIT 10
                    '''
    return query_upper2

df_upper_group2 = pd.read_sql(upper_group_by_2(),conn_nyc)
display(df_upper_group2)

Seems simple, yes?

But the string functions take on a bit more complexity when you are doing aggregations.

Let's look at a another example.

In [ ]:
def upper_compare():
    query_upper_compare = '''
                    SELECT distinct City, COUNT(*)
                    FROM DATA
                    WHERE UPPER(City) = 'ASTORIA'
                    GROUP BY City
                    ORDER BY City
                    '''
    return query_upper_compare

df_upper_compare = pd.read_sql(upper_compare(),conn_nyc)
display(df_upper_compare)

We can see that there are two possible spellings for this city, and SQLite considers them to be different.

Let's extend the CITY query from above, to include more than 5 rows.

In [ ]:
def simple_group_by_2():
    query_group_2 = '''
                    SELECT City, COUNT(*)
                    FROM DATA
                    GROUP BY City
                    ORDER BY COUNT(*) DESC
                    LIMIT 15
                    '''
    return query_group_2

df_simple_group_2 = pd.read_sql(simple_group_by_2(),conn_nyc)
display(df_simple_group_2)

### Note that the cities `Astoria`, `Jamaica`, and `Flushing` all have two different entries in the `City` column.

#### Students need to understand this behavior when grouping aggregations, and account for it.

#### The best way is to use a STRING MANIPULATION function, such as UPPER(), LOWER(), or COLLATE NOCASE.

Let's look at the two examples below.

In this first example:

See the two ways of handling the case-sensitive grouping. The first query uses the column aliases and the second uses the actual `UPPER()` function.

Either method is fine, and neither is better than the other.

In [ ]:
def upper_group_by():
# group by the column alias
    query_upper_group = '''
                    SELECT UPPER(City) AS CITY, COUNT(*)
                    FROM DATA
                    GROUP BY CITY
                    ORDER BY COUNT(*) DESC
                    LIMIT 10
                    '''
# group by the UPPER function
#     query_upper_group = '''
#                 SELECT UPPER(City) AS CITY, COUNT(*)
#                 FROM DATA
#                 GROUP BY UPPER(City)
#                 ORDER BY COUNT(*) DESC
#                 LIMIT 10
#                 '''

    return query_upper_group

df_upper_group = pd.read_sql(upper_group_by(),conn_nyc)
display(df_upper_group)

**Case-insensitive grouping: `COLLATE NOCASE`.** Another way to carry out the preceding query in a case-insensitive way is to add a `COLLATE NOCASE` qualifier to the `GROUP BY` clause.

The next example demonstrates this clause.

Note that the two query versions return slightly different results.

Take the city `Jamaica` for example. While each query returns 3,260 rows with this city, note that the two queries return different CityName values. There is not a rule for which it will return, in the second query, so students must be aware and understand how they want the data to appear in their results.

> Finally, the `COLLATE NOCASE` clause modifies the column next to which it appears. So if you are grouping by more than one key and want to be case-insensitive, you need to write, `... GROUP BY ColumnA COLLATE NOCASE, ColumnB COLLATE NOCASE ...`.

In [ ]:
def collate_group_by():
# group by the column alias
    query_collate_group = '''
                    SELECT UPPER(City) AS CityName, COUNT(*)
                    FROM DATA
                    GROUP BY City COLLATE NOCASE
                    ORDER BY COUNT(*) DESC
                    LIMIT 10
                    '''

#     query_collate_group = '''
#                 SELECT City AS CityName, COUNT(*)
#                 FROM DATA
#                 GROUP BY City COLLATE NOCASE
#                 ORDER BY COUNT(*) DESC
#                 LIMIT 10
#                 '''

    return query_collate_group

df_collate_group = pd.read_sql(collate_group_by(),conn_nyc)
display(df_collate_group)

### So what happens if we don't handle the case-sensitivity?

#### We can see from the examples that the City  differences give different results.

**So the takeaway is that we must ensure that we are correctly accounting for the data differences.**

Grouping in SQL is case-sensitive, so we must ensure that our code recognizes and deals with this.